In [1]:
import io
import gc
import fitz
import torch
import base64
import transformers
from PIL import Image
import pypdfium2 as pdfium
from dotenv import load_dotenv

from huggingface_hub.utils import enable_progress_bars
from colpali_engine.models import ColQwen2, ColQwen2Processor

from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend

c:\Users\UserAdmin\Documents\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import torch

print("=== CUDA GPU STATUS CHECK ===")
cuda_available = torch.cuda.is_available()
print(f"Is CUDA available in this notebook?: {cuda_available}")

if cuda_available:
    print(f"Active Graphics Card Name: {torch.cuda.get_device_name(0)}")
else:
    print("PyTorch cannot find your GPU.")


=== CUDA GPU STATUS CHECK ===
Is CUDA available in this notebook?: False
PyTorch cannot find your GPU.


In [9]:
load_dotenv()

True

In [8]:
pdf_path = r"C:\Users\UserAdmin\Documents\Multimodal-RAG\pdfs\government-personal-data-protection-policies-jul21.pdf"

In [10]:
model_name = "vidore/colqwen2-v1.0"
model = ColQwen2.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()

Loading weights: 100%|██████████| 392/392 [00:00<00:00, 3590.13it/s]
ColQwen2 LOAD REPORT from: vidore/colqwen2-v1.0
Key                                                                    | Status     | 
-----------------------------------------------------------------------+------------+-
base_model.language_model.model.custom_text_proj.lora_B.default.weight | UNEXPECTED | 
base_model.language_model.model.custom_text_proj.lora_A.default.weight | UNEXPECTED | 
custom_text_proj.lora_B.default.weight                                 | MISSING    | 
custom_text_proj.lora_A.default.weight                                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
processor = ColQwen2Processor.from_pretrained(
    model_name, 
    trust_remote_code=True
)

In [12]:
def get_coarse_embedding(embeddings_tensor):
    pooled = embeddings_tensor.mean(dim=1) # mean pooling 
    normalised = pooled / pooled.norm(dim=-1, keepdim=True) # L2 normalization 
    return normalised.squeeze().to(torch.float32).cpu()

In [13]:
def embed(pdf_path, model, processor):
    vector_index = []
    fitz_doc = fitz.open(pdf_path)
    for i in range(len(fitz_doc)):
        page = fitz_doc.load_page(i)
        pix = page.get_pixmap(dpi=150)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")

        inputs = processor.process_images(images=[img])
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            embeddings = model(**inputs)
            embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True) #L2 normalization 
            coarse_vector = get_coarse_embedding(embeddings).flatten().tolist()
            embeddings = embeddings.squeeze(0).to(torch.float32).cpu()

        vector_index.append({
            "page_number": i+1,
            "coarse_vector": coarse_vector,
            "page_embeddings": embeddings
        })
    return vector_index

        

    
    

In [14]:
vector_index = embed(pdf_path, model, processor)

In [15]:
vector_index[0]["page_embeddings"]

tensor([[ 9.7656e-03, -8.5938e-02, -6.6406e-02,  ...,  4.7119e-02,
          1.2512e-03,  1.1621e-01],
        [ 2.3804e-02, -1.0938e-01, -2.5757e-02,  ...,  7.1777e-02,
         -4.7363e-02, -1.4453e-01],
        [ 1.6022e-03, -9.9609e-02, -1.8677e-02,  ...,  7.1777e-02,
         -3.7109e-02, -1.3379e-01],
        ...,
        [ 4.3701e-02, -1.1182e-01, -6.3965e-02,  ...,  5.6641e-02,
          6.2256e-03,  7.9590e-02],
        [ 3.7109e-02, -1.1816e-01, -4.6875e-02,  ...,  4.0039e-02,
         -5.0537e-02, -1.2988e-01],
        [ 1.3672e-01, -7.0312e-02,  1.0073e-05,  ..., -3.1494e-02,
         -9.8145e-02, -5.6396e-02]])